In [ ]:
!rm -rf '/content/cg_nerf/'
!git clone https://github.com/LazyGH/cg_nerf.git
%cd /content/pj
%pip install -U imageio imageio-ffmpeg scipy matplotlib lpips configargparse tensorboard
!apt-get update
!apt-get install -y imagemagick

# NeRF

In [ ]:
%cd /content/pj
%pip install -U "tensorflow[and-cuda]" configargparse imageio imageio-ffmpeg matplotlib scipy lpips
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
%cd /content/pj/nerf
!bash download_example_data.sh

Train NeRF Lego

In [ ]:
%%time
%cd /content/pj/nerf
!python run_nerf.py --config course_configs/lego_t4_20min.txt

Train NeRF Fern

In [ ]:
%%time
%cd /content/pj/nerf
!python run_nerf.py --config course_configs/fern_t4_20min.txt

NeRF Evaluation

Trained Lego

In [ ]:
%cd /content/pj
!python project_tools/render_eval_nerf.py --config nerf/course_configs/lego_t4_20min.txt --output_dir result/nerf/lego_trained --source trained --train_time_minutes 20

Trained Fern

In [ ]:
%cd /content/pj
!python project_tools/render_eval_nerf.py --config nerf/course_configs/fern_t4_20min.txt --output_dir result/nerf/fern_trained --source trained --train_time_minutes 20

download paper pre-trained model

In [ ]:
%cd /content/pj/nerf
!bash download_example_weights.sh

Paper Lego

In [ ]:
%cd /content/pj
!python project_tools/render_eval_nerf.py --config nerf/course_configs/lego_t4_20min.txt --ckpt nef/logs/lego_example/model_fine_200000.npy --output_dir result/nerf/lego_pretrained --source pretrained

Paper Fern

In [ ]:
%cd /content/pj
!python project_tools/render_eval_nerf.py --config nerf/course_configs/fern_t4_30min.txt --ckpt nef/logs/lego_example/model_fine_200000.npy --output_dir result/nerf/fern_pretrained  --source pretrained

# DVGO

In [ ]:
%cd /content/pj
!pip install -U ninja tqdm opencv-python-headless imageio imageio-ffmpeg scipy lpips torch_efficient_distloss einops openmim mmcv-lite

import torch
import os
os.environ['TORCH'] = torch.__version__.split('+')[0]
os.environ['CUDA'] = 'cpu' if torch.version.cuda is None else f"cu{torch.version.cuda.replace('.', '')}"

# 4. Verify they were set
print("TORCH:", os.environ['TORCH'])
print("CUDA:", os.environ['CUDA'])

!pip install torch_scatter -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html

%cd /content/pj
!mkdir -p DirectVoxGO/data
!cp -r nerf/data/nerf_synthetic DirectVoxGO/data/
!cp -r nerf/data/nerf_llff_data DirectVoxGO/data/

%cd /content/pj/DirectVoxGO
from lib import dvgo
print('DVGO CUDA extension loaded successfully.')

Train DVGO lego

In [ ]:
%%time
%cd /content/pj/DirectVoxGO
python run.py --config configs/course/lego_t4_20min.py --render_test --dump_images --eval_ssim --eval_lpips_alex

Train DVGO fern

In [ ]:
%%time
%cd /content/pj/DirectVoxGO
python run.py --config configs/course/fern_t4_20min.py --render_test --dum p_images --eval_ssim --eval_lpips_alex

DVGO Evaluation

Lego

In [ ]:
%cd /content/pj
!python project_tools/render_eval_dvgo.py --config DirectVoxGO/configs/course/lego_t4_20min.py --output_dir result/dvgo/lego_trained --source traine--train_time_minutes 20

Fern

In [ ]:
%cd /content/pj
!python project_tools/render_eval_dvgo.py --config DirectVoxGO/configs/course/fern_t4_20min.py --output_dir result/dvgo/fern_trained --source trained --train_time_minutes 20

Summary quantitative Results

In [ ]:
%%bash
cd /content/pj
python project_tools/aggregate_metrics.py \
  result/nerf/lego_trained/metrics.json \
  result/nerf/fern_trained/metrics.json \
  result/nerf/lego_pretrained/metrics.json \
  result/nerf/fern_pretrained/metrics.json \
  result/dvgo/lego_trained/metrics.json \
  result/dvgo/fern_trained/metrics.json \
  --out_csv result/summary_metrics.csv

Save data

In [ ]:
from google.colab import drive
drive.mount('/content/drive/My Drive/Colab Notebooks/CG-pj')
!cp -r /content/pj/nerf/logs/* /content/drive/My Drive/Colab Notebooks/CG-pj/nerf/logs/
!cp -r /content/pj/DirectVoxGO/logs/* /content/drive/My Drive/Colab Notebooks/CG-pj/DirectVoxGO/logs/
!cp -r /content/pj/result /content/drive/My Drive/Colab Notebooks/CG-pj/result/

Tensorboard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/pj/nerf/logs/summaries

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/pj/DirectVoxGO/logs